In [ ]:
This notebook:
1. Loads and inspects the raw dataset
2. Performs exploratory data analysis
3. Checks missing values and data quality
4. Visualizes churn distribution and important patterns
5. Saves cleaned data for model training

Contribution:
- Data quality validation
- Class imbalance analysis
- Business-focused churn insights

In [3]:
pip install seaborn matplotlib pandas numpy jupyter notebook ipykernel

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
   ---------------------------------------- 0.0/139.8 kB ? eta -:--:--
   -- ------------------------------------- 10.2/139.8 kB ? eta -:--:--
   -------- ------------------------------ 30.7/139.8 kB 640.0 kB/s eta 0:00:01
   ------------------------- ------------- 92.2/139.8 kB 744.7 kB/s eta 0:00:01
   ------------------------- ------------- 92.2/139.8 kB 744.7 kB/s eta 0:00:01
   ------------------------- ------------- 92.2/139.8 kB 744.7 kB/s eta 0:00:01
   -------------------------------------- 139.8/139.8 kB 518.3 kB/s eta 0:00:00
   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---- ----------------------------------- 92.2/914.9 kB ? eta -:--:--
   ---- ----------------------------------- 92.2/914.9 kB ? eta -:--:--
   --------- ------------------------------ 225.3/914.9 kB 2.3 MB/s eta 0:00:01
   -------------- -------------------------


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\ASUS\Downloads\mini_pilot_project_v2\emoenv\Scripts\python.exe -m pip install --upgrade pip


In [9]:
%pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: C:\Users\ASUS\Downloads\mini_pilot_project_v2\emoenv\Scripts\python.exe -m pip install --upgrade pip


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:

# Paths

# This cell defines all project paths.
# We use an absolute project path to avoid Jupyter path errors.

from pathlib import Path

BASE_DIR = Path(r"C:\Users\ASUS\Downloads\RetainIQ")

RAW_DATA_PATH = BASE_DIR / "data" / "raw" / "telco_churn.csv"
PROCESSED_DATA_PATH = BASE_DIR / "data" / "processed" / "telco_churn_clean.csv"
FIGURES_DIR = BASE_DIR / "reports" / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Raw data path:", RAW_DATA_PATH)
print("Processed data path:", PROCESSED_DATA_PATH)
print("Figures directory:", FIGURES_DIR)

# Safety check to confirm the CSV file exists.
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {RAW_DATA_PATH}\n"
        "Please make sure telco_churn.csv is saved inside data/raw/"
    )

Base directory: C:\Users\ASUS\Downloads\RetainIQ
Raw data path: C:\Users\ASUS\Downloads\RetainIQ\data\raw\telco_churn.csv
Processed data path: C:\Users\ASUS\Downloads\RetainIQ\data\processed\telco_churn_clean.csv
Figures directory: C:\Users\ASUS\Downloads\RetainIQ\reports\figures


In [5]:

# Load Dataset

# The raw IBM Telco Customer Churn dataset is loaded from data/raw.

df = pd.read_csv(RAW_DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [6]:
#Dataset Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [7]:
#Missing Values

missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

Series([], dtype: int64)

In [ ]:
#Fix TotalCharges

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges after conversion:", df["TotalCharges"].isnull().sum())

df = df.dropna(subset=["TotalCharges"]).copy()

print("Shape after removing invalid TotalCharges rows:", df.shape)

In [ ]:
#Target Distribution

churn_counts = df["Churn"].value_counts()

plt.figure(figsize=(6, 4))
sns.barplot(x=churn_counts.index, y=churn_counts.values)
plt.title("Churn Class Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.tight_layout()

plt.savefig(FIGURES_DIR / "churn_class_distribution.png", dpi=300)
plt.show()

churn_counts

In [ ]:
#Churn Percentage

churn_percentage = df["Churn"].value_counts(normalize=True) * 100
churn_percentage

In [ ]:
print(f"No Churn: {churn_percentage['No']:.2f}%")
print(f"Churn: {churn_percentage['Yes']:.2f}%")

In [ ]:
#Churn by Contract Type

contract_churn = pd.crosstab(
    df["Contract"],
    df["Churn"],
    normalize="index"
) * 100

contract_churn.plot(kind="bar", figsize=(8, 5))
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Percentage")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(FIGURES_DIR / "churn_by_contract.png", dpi=300)
plt.show()

contract_churn

In [ ]:
#Churn by Internet Service

internet_churn = pd.crosstab(
    df["InternetService"],
    df["Churn"],
    normalize="index"
) * 100

internet_churn.plot(kind="bar", figsize=(8, 5))
plt.title("Churn Rate by Internet Service")
plt.xlabel("Internet Service")
plt.ylabel("Percentage")
plt.xticks(rotation=0)
plt.tight_layout()

plt.savefig(FIGURES_DIR / "churn_by_internet_service.png", dpi=300)
plt.show()

internet_churn

In [ ]:
#Monthly Charges vs Churn

plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges Distribution by Churn")
plt.tight_layout()

plt.savefig(FIGURES_DIR / "monthly_charges_by_churn.png", dpi=300)
plt.show()

In [ ]:
#Tenure vs Churn

plt.figure(figsize=(7, 5))
sns.boxplot(data=df, x="Churn", y="tenure")
plt.title("Customer Tenure Distribution by Churn")
plt.tight_layout()

plt.savefig(FIGURES_DIR / "tenure_by_churn.png", dpi=300)
plt.show()

In [ ]:
#Basic Data Quality Report

data_quality_report = {
    "total_rows": len(df),
    "total_columns": len(df.columns),
    "duplicate_rows": df.duplicated().sum(),
    "missing_values_total": df.isnull().sum().sum(),
    "target_column": "Churn",
    "churn_rate_percent": round((df["Churn"].eq("Yes").mean()) * 100, 2),
}

data_quality_report

In [ ]:
#Save Data Quality Report

quality_df = pd.DataFrame(
    list(data_quality_report.items()),
    columns=["Metric", "Value"]
)

quality_df.to_csv(BASE_DIR / "reports" / "data_quality_report.csv", index=False)

quality_df

In [ ]:
#Clean Dataset for Next Step

clean_df = df.copy()

clean_df = clean_df.drop(columns=["customerID"])

clean_df["Churn"] = clean_df["Churn"].map({
    "No": 0,
    "Yes": 1
})

clean_df.to_csv(PROCESSED_DATA_PATH, index=False)

print("Clean dataset saved to:", PROCESSED_DATA_PATH)
print("Shape:", clean_df.shape)

clean_df.head()